In [6]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("../balance_metrics")

import yaml
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


data_path = "../../experiment_data/balance_metrics/cross_layer_random.csv"

random_data = pd.read_csv(data_path)

with open('../across-blocks/layer_orders_cross_layer.yml', 'r') as f:
    layer_order = yaml.safe_load(f)
        
import matplotlib.pyplot as plt

import plotly.io as pio
pio.renderers.default = "vscode"

color_seq = px.colors.qualitative.Dark24
color_seq_train = px.colors.qualitative.Pastel
color_seq_val = px.colors.qualitative.Set1

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
full_data = pd.concat([random_data], ignore_index=True)
full_data = full_data.sort_values(["dataset"], ascending=True)

In [ ]:
resnet = full_data[(full_data["model"].str.contains("resnet")) & (full_data["split"] == "trainUval")].copy()

resnet["layer_idx"] = resnet["layer"].apply(lambda x: layer_order['resnet'][x]['idx'])
resnet["layer_names"] = resnet["layer"].apply(lambda x: layer_order['resnet'][x]['name'])
resnet.sort_values("layer_idx", inplace=True)

resnet_accs = resnet[(resnet["layer"] == "a1") & (resnet["dataset"].str.contains("-r"))][["dataset", "model", "train_acc", "val_acc", "sackin_index"]]
resnet_accs["random_prop"] = resnet_accs["dataset"].apply(lambda x: float(x.split("-r")[1]))
resnet_accs.sort_values("random_prop", inplace=True)

fig = make_subplots()
for i, row in resnet_accs.iterrows():
    color = color_seq[i % len(color_seq)]
    fig.add_trace(go.Scatter(x=[row["random_prop"], row["random_prop"]], y=[row["train_acc"], row["val_acc"]], mode="lines+markers", line = dict(color=color), name=f"{row['model']} ({row['random_prop']})"))
    
fig.update_layout(showlegend=False, margin=dict(l=0, r=0, t=0, b=0), width=600*2, height=300*2)
fig.update_xaxes(title_text="Randomness Proportion", nticks=21)
fig.update_yaxes(range=[0.0, 1.05], nticks=10, title_text="Accuracy")

In [36]:
penultimate_accs = full_data[(full_data["layer"] == "a1") & (full_data["dataset"].str.contains("-r")) & (full_data["split"] == "trainUval")][["dataset", "model", "train_acc", "val_acc", "sackin_index"]]
penultimate_accs["random_prop"] = penultimate_accs["dataset"].apply(lambda x: float(x.split("-r")[1]))
penultimate_accs.sort_values("random_prop", inplace=True)

fig = px.line(penultimate_accs, x="random_prop", y="sackin_index", color="model")

fig.update_layout(showlegend=True, margin=dict(l=0, r=0, t=0, b=0), width=600*2, height=300*2)
fig.update_xaxes(title_text="Randomness Proportion", nticks=21)
fig.update_yaxes(title_text="Sackin Index")
fig.show()

In [39]:
resnet["rprop"] = resnet["dataset"].apply(lambda x: float(x.split("-r")[-1]))
resnet2 = resnet[resnet['rprop'].isin([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])].copy()

fig = px.line(resnet2, x="layer_names", y="sackin_index", color="dataset", symbol="model", color_discrete_sequence=px.colors.qualitative.Plotly, title="ResNet Sackin")
fig.update_xaxes(tickangle=80, title_text="Layer")
fig.update_yaxes(title_text="Sackin Index")
fig.update_layout(showlegend=False, margin=dict(l=0, r=0, t=0, b=0), width=600*2, height=300*2)